In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd

path = "/kaggle/input/competitions/playground-series-s6e7"

train = pd.read_csv(f"{path}/train.csv")
test = pd.read_csv(f"{path}/test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv")

print(train.shape)
print(test.shape)

train.head()

In [ ]:
train.info()

In [ ]:
train["health_condition"].value_counts()

In [ ]:
X = train.drop(columns=["health_condition", "id"])
y = train["health_condition"]

X_test = test.drop(columns=["id"])

print(X.shape)
print(y.shape)
print(X_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)

## CatBoost Baseline

We train a CatBoost multiclass classifier because it can handle categorical features, missing values, and nonlinear relationships directly.

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report

categorical_columns = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

categorical_indices = [
    X_train.columns.get_loc(column)
    for column in categorical_columns
]

for column in categorical_columns:
    X_train[column] = X_train[column].fillna("Missing").astype(str)
    X_valid[column] = X_valid[column].fillna("Missing").astype(str)
    X_test[column] = X_test[column].fillna("Missing").astype(str)

In [ ]:
cat_model = CatBoostClassifier(
    iterations=700,
    depth=7,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=categorical_indices,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=80
)

In [ ]:
y_pred_cat = cat_model.predict(X_valid).ravel()

cat_score = balanced_accuracy_score(y_valid, y_pred_cat)

print("CatBoost Balanced Accuracy:", cat_score)
print(classification_report(y_valid, y_pred_cat))

## Train Final CatBoost Model and Create Submission

We retrain CatBoost using all labeled training data, then predict the health condition for Kaggle's test set.

In [ ]:
X_full = X.copy()

for column in categorical_columns:
    X_full[column] = X_full[column].fillna("Missing").astype(str)

final_cat_model = CatBoostClassifier(
    iterations=692,
    depth=7,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=100
)

final_cat_model.fit(
    X_full,
    y,
    cat_features=categorical_indices
)

test_predictions_cat = final_cat_model.predict(X_test).ravel()

submission_cat = sample_submission.copy()
submission_cat["health_condition"] = test_predictions_cat

submission_cat.to_csv(
    "submission_catboost.csv",
    index=False
)

submission_cat.head()

CatBoost baseline with native categorical handling, balanced class weights, and early-stopping-selected iterations. Validation balanced accuracy: 0.9499.